# Importing libraries
# Reading csv

In [1]:
import pandas as pd

df = pd.read_csv('paper_leaks.csv')

# 01. Fixing date format

In [2]:
# ── 1. fix date ──────────────────────────────────────────
df['date'] = pd.to_datetime(df['date'], format='mixed')
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# 02. Standardizing action

In [3]:
# ── 2. standardize action_taken into simple categories ───
# too many unique values — collapse into 4 clean buckets

def categorize_action(val):
    val = str(val).lower()
    if 'none' in val:
        return 'Nothing'
    
    has_cancel = 'cancel' in val
    has_arrest = 'arrest' in val
    has_conviction = 'convict' in val
    
    if has_conviction:
        return 'Justice'
    elif has_cancel and has_arrest:
        return 'Partial — Cancel + Arrest'
    elif has_arrest:
        return 'Partial — Arrest Only'
    elif has_cancel:
        return 'Partial — Cancel Only'
    else:
        return 'Partial — Probe Only'

df['action_clean'] = df['action_taken'].apply(categorize_action)
print(df['action_clean'].value_counts())

action_clean
Partial — Cancel + Arrest    45
Partial — Arrest Only        44
Partial — Cancel Only        12
Partial — Probe Only          5
Nothing                       4
Name: count, dtype: int64


# 03. Standardizing area

In [4]:
# ── 3. standardize area into state name only ─────────────
# too granular right now — "Rajasthan (Jaipur/Kota)" → "Rajasthan"

df['state'] = df['area'].str.extract(r'^([^(]+)').iloc[:, 0].str.strip()

# fix "All India" and "Bihar (Patna) / Jharkhand (arrest)" edge cases
df['state'] = df['state'].replace({
    'Bihar ': 'Bihar',
    'Bihar (Patna) / Jharkhand (arrest)': 'Bihar',
    'Delhi (NCT of Delhi)': 'Delhi',
    'Uttar Pradesh (Kanpur/Lucknow)': 'Uttar Pradesh',
    'Uttar Pradesh (Lucknow)': 'Uttar Pradesh',
    'Uttar Pradesh (Meerut)': 'Uttar Pradesh',
    'Uttar Pradesh (statewide)': 'Uttar Pradesh',
})

print("\nstate value counts:")
print(df['state'].value_counts())


state value counts:
state
All India                    16
Madhya Pradesh               13
Uttar Pradesh                12
Rajasthan                    11
Bihar                         7
Uttarakhand                   6
Maharashtra                   5
Gujarat                       5
Haryana                       5
Punjab                        4
Jharkhand                     4
Himachal Pradesh              3
Delhi                         3
Chhattisgarh                  3
Assam                         2
Karnataka                     2
Jammu & Kashmir               1
Manipur                       1
West Bengal                   1
Andaman & Nicobar Islands     1
Tamil Nadu                    1
Andhra Pradesh                1
Arunachal Pradesh             1
Telangana                     1
Odisha                        1
Name: count, dtype: int64


# 04. Fixing nulls

In [5]:
# ── 4. fill numeric nulls with 0 for counting purposes ───
# keep original columns, make filled versions for analysis
df['arrests_filled'] = df['arrests'].fillna(0)
df['convictions_filled'] = df['convictions'].fillna(0)
df['deaths_filled'] = df['linked_deaths'].fillna(0)

# 05. Saving clean csv

In [6]:
# ── 5. save cleaned file ─────────────────────────────────
df.to_csv('paper_leaks_cleaned.csv', index=False)

print("\nCleaned file saved.")
print(df.shape)
print(df['action_clean'].value_counts())


Cleaned file saved.
(110, 25)
action_clean
Partial — Cancel + Arrest    45
Partial — Arrest Only        44
Partial — Cancel Only        12
Partial — Probe Only          5
Nothing                       4
Name: count, dtype: int64
